In [1]:
import os
import json
import pandas as pd

# Load runtime config (from Notebook 10)
CONFIG_PATH = "config/runtime_config.json"

if not os.path.exists(CONFIG_PATH):
    raise FileNotFoundError("runtime_config.json not found — run Notebook 10")

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

INPUT_PATH = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/data/real/df_working.csv"

OUTPUT_CSV_PATH = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/reports/profiling_reports/dataset_profile.csv"
OUTPUT_JSON_PATH = "/media/ausaafkhan/New Volume1/project_exhibition/broken_dataset_repair/reports/profiling_reports/dataset_profile.json"

In [2]:
# 🔹 CELL 2 — Load Dataset (FAIL-FAST)

if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"Input file not found at: {INPUT_PATH}")

df_working = pd.read_csv(INPUT_PATH)

if df_working.shape[0] == 0:
    raise ValueError("Dataset is empty (0 rows)")

if df_working.columns.isnull().any():
    raise ValueError("Invalid column names detected (null column name)")

if len(set(df_working.columns)) != len(df_working.columns):
    raise ValueError("Duplicate column names detected")

In [3]:
# 🔹 CELL 3 — Deterministic Mode Function

def deterministic_mode(series):
    value_counts = series.dropna().value_counts()

    if value_counts.empty:
        return None, 0

    max_count = value_counts.max()
    candidates = value_counts[value_counts == max_count].index.tolist()
    candidates = sorted([str(c) for c in candidates])

    return candidates[0], int(max_count)

In [4]:
# 🔹 CELL 4 — Profiling Loop

profile_rows = []
total_rows = len(df_working)

for col in df_working.columns:
    try:
        series = df_working[col]
        dtype = str(series.dtype)

        non_null_count = int(series.notna().sum())
        null_count = int(series.isna().sum())
        null_percentage = null_count / total_rows

        unique_count = int(series.nunique(dropna=True))
        duplicate_count = non_null_count - unique_count
        duplicate_percentage = duplicate_count / total_rows

        mode_value, mode_count = deterministic_mode(series)

        row = {
            "column_name": col,
            "dtype": dtype,
            "total_rows": total_rows,
            "non_null_count": non_null_count,
            "null_count": null_count,
            "null_percentage": null_percentage,
            "unique_count": unique_count,
            "duplicate_count": duplicate_count,
            "duplicate_percentage": duplicate_percentage,
            "most_frequent_value": mode_value,
            "most_frequent_count": mode_count,
            "mean": None,
            "median": None,
            "std": None,
            "min": None,
            "max": None,
            "percentile_25": None,
            "percentile_75": None,
            "avg_length": None,
            "min_length": None,
            "max_length": None,
            "zero_count": None,
            "negative_count": None,
            "empty_string_count": None,
            "whitespace_string_count": None,
        }

        if pd.api.types.is_numeric_dtype(series):
            clean_series = series.dropna()

            if len(clean_series) > 0:
                row.update({
                    "mean": float(clean_series.mean()),
                    "median": float(clean_series.median()),
                    "std": float(clean_series.std()),
                    "min": float(clean_series.min()),
                    "max": float(clean_series.max()),
                    "percentile_25": float(clean_series.quantile(0.25)),
                    "percentile_75": float(clean_series.quantile(0.75)),
                    "zero_count": int((series == 0).sum()),
                    "negative_count": int((series < 0).sum()),
                })
            else:
                row.update({
                    "mean": 0.0,
                    "std": 0.0,
                    "zero_count": 0,
                    "negative_count": 0,
                })

        if pd.api.types.is_object_dtype(series):
            clean_series = series.dropna().astype(str)

            if len(clean_series) > 0:
                lengths = clean_series.map(len)

                row.update({
                    "avg_length": float(lengths.mean()),
                    "min_length": int(lengths.min()),
                    "max_length": int(lengths.max()),
                    "empty_string_count": int((series == "").sum()),
                    "whitespace_string_count": int(series.apply(
                        lambda x: isinstance(x, str) and x.strip() == "" and x != ""
                    ).sum())
                })
            else:
                row.update({
                    "empty_string_count": 0,
                    "whitespace_string_count": 0
                })

        profile_rows.append(row)

    except Exception as e:
        raise RuntimeError(f"Profiling failed for column: {col} | Error: {str(e)}")

In [5]:
# 🔹 CELL 5 — Create dataset_profile

dataset_profile = pd.DataFrame(profile_rows)

if len(dataset_profile) != len(df_working.columns):
    raise ValueError("Mismatch: profiling output rows != number of columns")

In [6]:
# 🔹 CELL 6 — Enforce Required Schema + Feature Completion (STRICT)

REQUIRED_COLUMNS = [
    "column_name",
    "row_count",
    "unique_count",
    "unique_ratio",
    "null_ratio",
    "std",
    "mean",
    "dtype"
]

base_required = ["column_name", "dtype", "total_rows", "unique_count"]

for col in base_required:
    if col not in dataset_profile.columns:
        raise ValueError(f"Missing required base column: {col}")

# row_count
dataset_profile["row_count"] = dataset_profile["total_rows"]

# unique_ratio
dataset_profile["unique_ratio"] = dataset_profile["unique_count"] / dataset_profile["row_count"]

# null_ratio
if "null_count" in dataset_profile.columns:
    dataset_profile["null_ratio"] = dataset_profile["null_count"] / dataset_profile["row_count"]
elif "null_percentage" in dataset_profile.columns:
    dataset_profile["null_ratio"] = dataset_profile["null_percentage"]
else:
    raise ValueError("Missing null information to compute null_ratio")

# mean/std fallback for non-numeric
dataset_profile["mean"] = dataset_profile["mean"].fillna(0.0)
dataset_profile["std"] = dataset_profile["std"].fillna(0.0)

# VALIDATION
for col in REQUIRED_COLUMNS:
    if col not in dataset_profile.columns:
        raise ValueError(f"Missing required column after enrichment: {col}")

if dataset_profile[REQUIRED_COLUMNS].isnull().any().any():
    raise ValueError("Null values detected in required columns")

if not ((dataset_profile["null_ratio"] >= 0) & (dataset_profile["null_ratio"] <= 1)).all():
    raise ValueError("null_ratio out of bounds [0,1]")

if not ((dataset_profile["unique_ratio"] >= 0) & (dataset_profile["unique_ratio"] <= 1)).all():
    raise ValueError("unique_ratio out of bounds [0,1]")

In [7]:
# 🔹 CELL 7 — Save Outputs

os.makedirs(os.path.dirname(OUTPUT_CSV_PATH), exist_ok=True)

dataset_profile.to_csv(OUTPUT_CSV_PATH, index=False)

with open(OUTPUT_JSON_PATH, "w") as f:
    json.dump(dataset_profile.to_dict(orient="records"), f, indent=4)

print("✅ Dataset profiling completed successfully")

✅ Dataset profiling completed successfully
